# **1. Scaled Dot-Product Attention**
수식 :
$\text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right)V$

### 1) 예제 입력 데이터 정의

Scaled Dot-Product Attention의 설명을 위해 예제 데이터를 생성합니다.

`I love this movie` 문장을 4차원 임베딩 벡터, 4개 시퀀스 예제로 임의의 데이터로 작성합니다.

설명을 위해 가중치가 없는 데이터셋을 설정합니다.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# Q는 단어 love 임베딩 벡터라고 가정합니다.
Q = torch.tensor([0.1, 0.3, 0.7, 0.9]) # shape: (4,)

K = torch.tensor([
    [0.3, 0.2, 0.8, 0.6], # I
    [0.1, 0.3, 0.7, 0.9], # love
    [0.6, 0.4, 0.5, 0.8], # this
    [0.7, 0.1, 0.4, 0.7], # movie
])
V = K
K

tensor([[0.3000, 0.2000, 0.8000, 0.6000],
        [0.1000, 0.3000, 0.7000, 0.9000],
        [0.6000, 0.4000, 0.5000, 0.8000],
        [0.7000, 0.1000, 0.4000, 0.7000]])

### 2) Scaled Dot-Product Attention 클래스

**`d_head`:**
*   Query/Key 벡터의 차원 수입니다.
*   여기서는 Q와 K의 차원이 4이므로 `d_head = 4`입니다.
*   Scaled Dot-Product Attention에서는 attention score를 `sqrt(d_head)`로 나눕니다.
*   이렇게 하는 이유는 차원이 커질수록 내적값이 커져 softmax가 너무 뾰족해지는 문제를 줄이기 위해서입니다.

**`Q`:**
*   Query 벡터 또는 Query 행렬입니다.
*   예제에서는 `love` 토큰의 Query 하나만 사용하므로 shape는 `(1, 4)`입니다.

**`K`:**
*   Key 행렬입니다.
*   shape는 `(4, 4)`입니다.
*   4개의 토큰이 있고, 각 토큰의 Key 차원이 4입니다.

**`V`:**
*   Value 행렬입니다.
*   shape는 `(4, 4)`입니다.
*   4개의 토큰이 있고, 각 토큰의 Value 차원이 4입니다.

**`attn_mask`:**
*   필요할 경우 특정 위치를 보지 못하게 막는 mask입니다.
*   예를 들어 Decoder의 causal mask에서 미래 토큰을 가릴 때 사용합니다.

In [2]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_head):
        super().__init__()
        self.d_head = d_head

    def forward(self, Q, K, V, attn_mask=None):
        # ------------------------------------------------------------
        # 1) Attention Score 계산
        # ------------------------------------------------------------
        # Query와 Key의 관련성을 내적으로 계산합니다.
        attention_scores = Q @ K.transpose(-2, -1)

        # ------------------------------------------------------------
        # 2) Mask 적용
        # ------------------------------------------------------------
        # attn_mask가 있다면 mask 값이 0인 위치를 -inf로 바꿉니다.
        # softmax(-inf)는 0이 되므로, 해당 위치는 attention을 받지 않게 됩니다.
        if attn_mask is not None:
            attention_scores = attention_scores.masked_fill(attn_mask == 0, float("-inf"))

        # ------------------------------------------------------------
        # 3) Scaling
        # ------------------------------------------------------------
        # Attention Score를 sqrt(d_head)로 나눕니다.
        scaled_scores = attention_scores / math.sqrt(self.d_head)

        # ------------------------------------------------------------
        # 4) Softmax로 Attention Weight 계산
        # ------------------------------------------------------------
        # scaled score를 softmax에 넣어 전체 합이 1인 가중치로 변환합니다.
        # attention_weights의 의미:
        # love Query가 각 토큰의 Value를 얼마나 참고할지 나타내는 비율입니다.
        attention_weights = F.softmax(scaled_scores, dim=-1)

        # ------------------------------------------------------------
        # 5) Context Vector 계산
        # ------------------------------------------------------------
        # Attention Weight를 Value에 곱해 가중합합니다.
        context_vector = attention_weights @ V
        return context_vector, attention_scores, scaled_scores, attention_weights

### 3) 실행

위에서 정의한 Scaled Dot-Product Attention 클래스를 사용하여 `Q_love`, `K`, `V`를 입력으로 실행한 결과는 다음과 같습니다.

#### 결과 분석:

*   **1. Attention Scores**: `Q_love`와 `K`의 내적 결과입니다. `tensor([[1.19, 1.4, 1.25, 1.01]])` 이 값들은 'love' Query가 각 Key(I, love, this, movie)와 얼마나 유사한지를 나타냅니다. 'love' 토큰의 Key와 가장 높은 관련성을 보입니다 (1.4).

*   **2. Scaled Scores**: Attention Scores를 `sqrt(d_head)` (여기서는 `sqrt(4) = 2`)로 나눈 값입니다. `tensor([[0.595, 0.7, 0.625, 0.505]])`. 이 스케일링은 소프트맥스 함수의 기울기 소실을 방지하는 데 도움을 줍니다.

*   **3. Attention Weights**: Scaled Scores에 소프트맥스 함수를 적용한 결과입니다. `tensor([[0.2466, 0.2739, 0.2541, 0.2254]])`. 이 값들은 각 Value 토큰이 최종 Context Vector에 얼마나 기여할지 나타내는 가중치이며, 모든 가중치의 합은 1입니다. 'love' 토큰의 Value에 가장 높은 가중치(0.2739)가 부여되었습니다.

*   **4. Context Vector, 즉 Z_love**: Attention Weights를 각 Value 토큰에 곱하여 합산한 최종 결과입니다. `tensor([[0.4116, 0.2557, 0.6062, 0.7555]])`. 이 벡터는 'love' Query에 대한 문맥적 정보가 요약된 표현입니다.

In [3]:
# Q는 현재 shape가 (4,)입니다.
# 행렬곱 결과를 그림처럼 (1, 4)로 보기 위해 (1, 4) 형태로 바꿉니다.
Q_love = Q.reshape(1, 4)

# d_head는 Query/Key 벡터의 차원입니다.
# 여기서는 K의 마지막 차원이 4입니다.
d_head = K.shape[-1]

attention = ScaledDotProductAttention(d_head=d_head)

context_vector, attention_scores, scaled_scores, attention_weights = attention(Q_love, K, V)


# ============================================================
# 4. 결과 확인
# ============================================================

print("Q_love shape:", Q_love.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print()

print("1. Attention Scores")
print(attention_scores)
print()

print("2. Scaled Scores")
print(scaled_scores)
print()

print("3. Attention Weights")
print(attention_weights)
print("sum:", attention_weights.sum(dim=-1))
print()

print("4. Context Vector, 즉 Z_love")
print(context_vector)

Q_love shape: torch.Size([1, 4])
K shape: torch.Size([4, 4])
V shape: torch.Size([4, 4])

1. Attention Scores
tensor([[1.1900, 1.4000, 1.2500, 1.0100]])

2. Scaled Scores
tensor([[0.5950, 0.7000, 0.6250, 0.5050]])

3. Attention Weights
tensor([[0.2466, 0.2739, 0.2541, 0.2254]])
sum: tensor([1.])

4. Context Vector, 즉 Z_love
tensor([[0.4116, 0.2557, 0.6062, 0.7555]])
